# Pipeline Modular de Fine-Tuning — Modelo DSCNN (4 Classes de Expressão Facial)

Notebook **100% autocontido** para fine-tuning progressivo do modelo **DSCNN** no Google Colab, lendo o checkpoint pré-treinado (`.pt`/`.pth`) e o dataset diretamente do **Google Drive** (`/content/drive/MyDrive/...`).

- **Classes Alvo (4 classes fixas)**: `0: angry`, `1: happy`, `2: neutral`, `3: sad` (descarta automaticamente `fear`, `disgust`, `surprise`, `contempt`).
- **Fase 1 (Warmup)**: Congela os primeiros **70%** dos blocos extratores (`features[0..6]`) e treina apenas o bloco final (`features[7..9]`) e o classificador linear (`head`).
- **Fase 2 (Full Fine-Tuning)**: Descongela toda a rede com **taxas de aprendizado discriminativas** (`lr_backbone = 0.1 * lr_head`), `AdamW(weight_decay=1e-2)`, `CrossEntropyLoss(label_smoothing=0.1)` e `CosineAnnealingLR`.
- **Validação e Avaliação com TTA**: Predição com Test-Time Augmentation (original + flip horizontal), salvamento automático do melhor checkpoint por **acurácia balanceada**, Matriz de Confusão e Relatório de Classificação (`Precision`, `Recall`, `F1-Score`).

In [ ]:
# Imports e utilitários de ambiente usados pelas demais células.
from __future__ import annotations
import ast
import csv
import json
import logging
import math
import pickle
import random
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    warnings.warn("matplotlib ausente; gráficos serão omitidos.", RuntimeWarning)
logger = logging.getLogger("dscnn_finetune")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


# Define as seeds aleatórias para tornar os resultados reproduzíveis.
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Detecta CPU ou GPU e informa o dispositivo ativo.
def get_runtime_device() -> torch.device:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch {torch.__version__} | device={device}")
    if device.type == "cuda":
        print("GPU:", torch.cuda.get_device_name(device))
    return device


RUNTIME_DEVICE = get_runtime_device()
seed_everything(42)


## Seção 1 — Configuração (`FineTuneConfig`)

Configure caminhos, formato dos dados, arquitetura e hiperparâmetros. A ordem dos quatro rótulos é fixa.

In [ ]:
# Classes canônicas e configuração do treinamento.
TARGET_CLASSES = ("angry", "happy", "neutral", "sad")
CLASS_TO_IDX = {name: i for i, name in enumerate(TARGET_CLASSES)}
IDX_TO_CLASS = {i: name for name, i in CLASS_TO_IDX.items()}
NUM_CLASSES = 4
DatasetFormat = Literal["auto", "csv", "imagefolder"]
LabelScheme = Literal["auto", "target", "fer", "rafdb", "affectnet"]


# Agrupa caminhos, opções e hiperparâmetros do fine-tuning.
@dataclass
class FineTuneConfig:
    pretrained_checkpoint_path: str | Path | None = "pretrained_dscnn.pt"
    train_data_path: str | Path = "data/train"
    val_data_path: str | Path | None = "data/val"
    output_checkpoint_path: str | Path = "best_dscnn_finetuned.pt"
    history_json_path: str | Path = "finetune_history.json"
    dataset_format: DatasetFormat = "auto"
    label_scheme: LabelScheme = "auto"
    csv_image_root: str | Path | None = None
    csv_usage_train: str | None = "Training"
    csv_usage_val: str | None = "PublicTest"
    image_size: int = 48
    width_multiplier: float = 1.0
    num_classes: int = 4
    batch_size: int = 64
    warmup_epochs: int = 5
    total_epochs: int = 30
    freeze_ratio: float = 0.7
    lr_head: float = 0.0003
    lr_backbone_factor: float = 0.1
    min_lr: float = 1e-06
    weight_decay: float = 0.01
    label_smoothing: float = 0.1
    grad_clip_norm: float | None = 1.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp: bool = True
    num_workers: int = 2
    pin_memory: bool = True
    seed: int = 42

    @property
    def lr_backbone(self) -> float:
        return self.lr_head * self.lr_backbone_factor

    def to_dict(self) -> dict[str, Any]:

        # Converte caminhos e contêineres em valores serializáveis.
        def clean(x):
            if isinstance(x, Path):
                return str(x)
            if isinstance(x, dict):
                return {k: clean(v) for k, v in x.items()}
            if isinstance(x, (tuple, list)):
                return [clean(v) for v in x]
            return x

        return clean(asdict(self))

    # Verifica limites e opções essenciais da configuração.
    def validate(self) -> None:
        if self.num_classes != 4:
            raise ValueError("O notebook exige quatro classes.")
        if self.total_epochs < 1 or not 0 <= self.warmup_epochs < self.total_epochs:
            raise ValueError("Épocas inválidas.")
        if self.image_size < 16 or self.batch_size < 1:
            raise ValueError("image_size >= 16 e batch_size >= 1.")
        if not 0 < self.freeze_ratio <= 1:
            raise ValueError("freeze_ratio deve estar em (0, 1].")


config = FineTuneConfig()
config.validate()
print(CLASS_TO_IDX, "| LR backbone:", config.lr_backbone)


## Seção 2 — Modelo DSCNN e pesos pré-treinados

A arquitetura tem dez blocos (`features.0` a `features.9`), pooling global e cabeça linear. O warmup congela `features[0:7]`, mantendo BatchNorm congelado; a fase completa descongela tudo. O loader reconhece `model_state_dict`, `state_dict`, prefixos DataParallel/wrapper e cabeças `fc`/`classifier`.

In [ ]:
# Blocos convolucionais, arquitetura DSCNN e carregamento de pesos.
# Monta um bloco convolucional com BatchNorm e ReLU6.
def _conv(in_channels, out_channels, kernel=3, stride=1, groups=1):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, stride, kernel // 2, groups=groups, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU6(inplace=True),
    )


# Encadeia convoluções depthwise e pointwise.
class _DepthwiseSeparable(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, pointwise_kernel=1):
        super().__init__()
        self.depthwise = _conv(in_channels, in_channels, 3, stride, in_channels)
        self.pointwise = _conv(in_channels, out_channels, pointwise_kernel)

    # Aplica convolução depthwise seguida da pointwise.
    def forward(self, x):
        return self.pointwise(self.depthwise(x))


# Define a rede DSCNN com backbone e cabeça de quatro classes.
class DSCNN(nn.Module):
    def __init__(self, num_classes=4, width_multiplier=1.0, pointwise_kernel=1):
        super().__init__()
        if width_multiplier <= 0 or pointwise_kernel not in (1, 3):
            raise ValueError("Parâmetros DSCNN inválidos.")
        c = lambda n: max(1, int(n * width_multiplier))
        self.features = nn.Sequential(
            _conv(1, c(32), 3, 2),
            _DepthwiseSeparable(c(32), c(64), 1, pointwise_kernel),
            _DepthwiseSeparable(c(64), c(128), 2, pointwise_kernel),
            _DepthwiseSeparable(c(128), c(128), 1, pointwise_kernel),
            _DepthwiseSeparable(c(128), c(256), 2, pointwise_kernel),
            _DepthwiseSeparable(c(256), c(256), 1, pointwise_kernel),
            _DepthwiseSeparable(c(256), c(256), 2, pointwise_kernel),
            _DepthwiseSeparable(c(256), c(256), 1, pointwise_kernel),
            _DepthwiseSeparable(c(256), c(256), 1, pointwise_kernel),
            _DepthwiseSeparable(c(256), c(256), 2, pointwise_kernel),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten(start_dim=1)
        self.head = nn.Linear(c(256), num_classes)
        self._frozen_cutoff = 0

    # Calcula os logits a partir das imagens de entrada.
    def forward(self, x):
        return self.head(self.flatten(self.pool(self.features(x))))

    # Congela os primeiros blocos do backbone e suas BatchNorm.
    def freeze_early_blocks(self, freeze_ratio=0.7):
        if not 0 <= freeze_ratio <= 1:
            raise ValueError("freeze_ratio deve estar entre 0 e 1.")
        self._frozen_cutoff = int(len(self.features) * freeze_ratio)
        for i, block in enumerate(self.features):
            trainable = i >= self._frozen_cutoff
            for p in block.parameters():
                p.requires_grad = trainable
            block.train(trainable)
        for p in self.head.parameters():
            p.requires_grad = True
        return self._frozen_cutoff

    # Mantém blocos congelados em modo de avaliação.
    def train(self, mode=True):
        super().train(mode)
        if mode and self._frozen_cutoff:
            for block in self.features[: self._frozen_cutoff]:
                block.eval()
        return self

    # Libera os parâmetros de toda a rede para treinamento.
    def unfreeze_all(self):
        self._frozen_cutoff = 0
        for p in self.parameters():
            p.requires_grad = True

    # Separa os parâmetros em grupos com taxas de aprendizado próprias.
    def get_param_groups(self, lr_head, lr_backbone_factor=0.1, weight_decay=0.01, phase="finetune"):
        if phase == "warmup":
            groups = [
                {
                    "name": "backbone_tail",
                    "params": [p for p in self.features.parameters() if p.requires_grad],
                    "lr": lr_head,
                    "weight_decay": weight_decay,
                },
                {
                    "name": "head",
                    "params": [p for p in self.head.parameters() if p.requires_grad],
                    "lr": lr_head,
                    "weight_decay": weight_decay,
                },
            ]
            return [g for g in groups if g["params"]]
        if phase != "finetune":
            raise ValueError(f"Fase desconhecida: {phase}")
        return [
            {
                "name": "backbone",
                "params": list(self.features.parameters()),
                "lr": lr_head * lr_backbone_factor,
                "weight_decay": weight_decay,
            },
            {"name": "head", "params": list(self.head.parameters()), "lr": lr_head, "weight_decay": weight_decay},
        ]


# Reconstrói a DSCNN e carrega pesos de um checkpoint.
def load_pretrained_dscnn(checkpoint_path, *, num_classes=4, width_multiplier=1.0, device="cpu", strict=True):
    state = None
    if checkpoint_path is not None:
        path = Path(checkpoint_path).expanduser()
        if not path.is_file():
            if strict:
                raise FileNotFoundError(f"Checkpoint não encontrado: {path}")
        else:
            try:
                ckpt = torch.load(path, map_location="cpu", weights_only=True)
            except (TypeError, pickle.UnpicklingError):
                ckpt = torch.load(path, map_location="cpu", weights_only=False)
            if isinstance(ckpt, dict) and isinstance(ckpt.get("model_state_dict"), dict):
                state = ckpt["model_state_dict"]
            elif isinstance(ckpt, dict) and isinstance(ckpt.get("state_dict"), dict):
                state = ckpt["state_dict"]
            elif isinstance(ckpt, dict):
                state = ckpt
            else:
                raise TypeError("Checkpoint inválido: esperado state_dict PyTorch.")
            clean = {}
            for old, value in state.items():
                key = str(old)
                while key.startswith(("module.", "model.")):
                    key = key.split(".", 1)[1]
                if key.startswith("fc."):
                    key = "head." + key[3:]
                if key.startswith("classifier."):
                    key = "head." + key[len("classifier.") :]
                clean[key] = value
            state = clean
    kernel = 1
    if state is not None:
        weight = state.get("features.1.pointwise.0.weight")
        if weight is not None and weight.ndim == 4:
            kernel = int(weight.shape[-1])
    model = DSCNN(num_classes, width_multiplier, kernel)
    if state is not None:
        model.load_state_dict(state, strict=strict)
    return model.to(device)


# Cria grupos AdamW usando as taxas da configuração.
def build_discriminative_param_groups(model, config, phase):
    return model.get_param_groups(config.lr_head, config.lr_backbone_factor, config.weight_decay, phase)


_model_probe = DSCNN(width_multiplier=0.125)
assert _model_probe.freeze_early_blocks(0.7) == 7
assert all((not p.requires_grad for b in _model_probe.features[:7] for p in b.parameters()))
assert math.isclose(_model_probe.get_param_groups(0.0003, 0.1)[0]["lr"], 3e-05, rel_tol=1e-12)
print("DSCNN: 10 blocos; congelamento 70% e LR discriminativa verificados.")


## Seção 3 — Dataset, remapeamento e transforms

Aceita diretórios ImageFolder, lista `list_patition_label.txt` do RAF-DB e CSVs com `pixels` ou caminho de imagem. Os esquemas inteiros `target`, `fer`, `rafdb` e `affectnet` mapeiam para `angry/happy/neutral/sad`; as outras emoções são excluídas. Transformações de treino incluem crop, flip, rotação e erasing; validação é determinística. Imagens corrompidas são registradas e substituídas pelo próximo item válido.

In [ ]:
# Remapeamento de emoções, transforms e leitura dos datasets.
STRING_LABEL_MAP = {
    "angry": 0,
    "anger": 0,
    "raiva": 0,
    "happy": 1,
    "happiness": 1,
    "feliz": 1,
    "alegria": 1,
    "joy": 1,
    "neutral": 2,
    "neutro": 2,
    "normal": 2,
    "sad": 3,
    "sadness": 3,
    "triste": 3,
    "tristeza": 3,
}
SCHEME_INT_MAPS = {
    "target": {0: 0, 1: 1, 2: 2, 3: 3},
    "fer": {0: 0, 3: 1, 6: 2, 4: 3},
    "rafdb": {6: 0, 4: 1, 7: 2, 5: 3},
    "affectnet": {6: 0, 1: 1, 0: 2, 2: 3},
}
DISCARDED_LABELS = {
    "fear",
    "afraid",
    "medo",
    "disgust",
    "nojo",
    "surprise",
    "surprised",
    "surpresa",
    "contempt",
    "desprezo",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# Remapeia rótulos de origem para as quatro classes-alvo.
def resolve_target_label(raw_label: int | str, scheme="auto") -> int | None:
    if scheme not in ("auto", *SCHEME_INT_MAPS):
        raise ValueError(f"Esquema inválido: {scheme}")
    if isinstance(raw_label, (int, np.integer)):
        return SCHEME_INT_MAPS["target" if scheme == "auto" else scheme].get(int(raw_label))
    text = re.sub("^[0-9]+[_\\- :]+", "", str(raw_label).strip().lower()).replace(" ", "_")
    if text in STRING_LABEL_MAP:
        return STRING_LABEL_MAP[text]
    if text in DISCARDED_LABELS:
        return None
    if text.isdigit():
        return SCHEME_INT_MAPS["target" if scheme == "auto" else scheme].get(int(text))
    return None


# Prepara imagens grayscale e aplica augmentations de treino.
def build_transforms(image_size=48, is_train=True):
    ops = [transforms.Grayscale(1), transforms.Resize((image_size, image_size))]
    if is_train:
        ops += [
            transforms.RandomCrop(image_size, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
        ]
    ops += [transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]
    if is_train:
        ops.append(transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value=0.0))
    return transforms.Compose(ops)


# Lê imagens e rótulos de pastas ou arquivos CSV.
class EmotionFinetuneDataset(Dataset):
    def __init__(
        self,
        data_path,
        *,
        transform=None,
        dataset_format="auto",
        label_scheme="auto",
        image_root=None,
        usage_filter=None,
        image_size=48,
    ):
        self.data_path = Path(data_path).expanduser()
        self.transform = transform or build_transforms(image_size, False)
        self.label_scheme = label_scheme
        self.image_root = Path(image_root).expanduser() if image_root else None
        self.usage_filter = usage_filter
        self.records = []
        self.discarded_count = 0
        self.corrupted_indices = set()
        fmt = dataset_format
        if fmt == "auto":
            fmt = "csv" if self.data_path.is_file() and self.data_path.suffix.lower() == ".csv" else "imagefolder"
        if fmt == "csv":
            self._read_csv()
        elif fmt == "imagefolder":
            self._read_directory()
        else:
            raise ValueError(f"Formato inválido: {fmt}")
        if not self.records:
            raise ValueError(f"Nenhuma amostra válida em {self.data_path}")

    # Filtra rótulos fora do escopo e registra amostras válidas.
    def _append(self, record, raw):
        label = resolve_target_label(raw, self.label_scheme)
        if label is None:
            self.discarded_count += 1
        else:
            record["label"] = label
            self.records.append(record)

    # Indexa imagens por pasta ou pela lista de anotações RAF-DB.
    def _read_directory(self):
        if not self.data_path.is_dir():
            raise FileNotFoundError(self.data_path)
        listing = self.data_path / "list_patition_label.txt"
        if listing.is_file():
            roots = [
                self.data_path,
                self.data_path / "Image",
                self.data_path / "aligned",
                self.data_path / "DATASET" / "train",
            ]
            for line in listing.read_text(encoding="utf-8", errors="replace").splitlines():
                parts = line.split()
                if len(parts) >= 2:
                    name, label = (parts[0], parts[-1])
                    path = next((root / name for root in roots if (root / name).is_file()), roots[0] / name)
                    self._append({"kind": "path", "path": path}, label)
            return
        for folder in sorted((p for p in self.data_path.iterdir() if p.is_dir())):
            for path in sorted((p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)):
                self._append({"kind": "path", "path": path}, folder.name)

    # Lê pixels ou caminhos de imagem de um CSV.
    def _read_csv(self):
        with self.data_path.open(encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            cols = {x.lower().strip(): x for x in reader.fieldnames or []}
            label_col = next(
                (cols[x] for x in ("emotion", "label", "expression", "target", "class") if x in cols), None
            )
            pixel_col = cols.get("pixels")
            path_col = next(
                (cols[x] for x in ("image_path", "path", "file", "filename", "subdirectory_filepath") if x in cols),
                None,
            )
            usage_col = next((cols[x] for x in ("usage", "split", "partition", "set") if x in cols), None)
            if not label_col or (not pixel_col and (not path_col)):
                raise ValueError("CSV precisa de rótulo e pixels ou caminho.")
            for row in reader:
                if (
                    self.usage_filter
                    and usage_col
                    and (row.get(usage_col, "").strip().lower() != self.usage_filter.lower())
                ):
                    continue
                if pixel_col:
                    record = {"kind": "pixels", "pixels": row.get(pixel_col, "")}
                else:
                    raw = Path(row.get(path_col, "").strip())
                    base = self.image_root or self.data_path.parent
                    record = {"kind": "path", "path": raw if raw.is_absolute() else base / raw}
                self._append(record, row.get(label_col, ""))

    # Retorna o número de amostras válidas indexadas.
    def __len__(self):
        return len(self.records)

    # Abre uma imagem ou reconstrói pixels inline do CSV.
    def _image(self, record):
        if record["kind"] == "path":
            with Image.open(record["path"]) as im:
                return im.convert("RGB")
        text = str(record["pixels"]).strip()
        try:
            values = np.fromstring(text, sep=" ", dtype=np.uint8)
        except ValueError:
            values = np.array([], dtype=np.uint8)
        if not values.size:
            values = np.asarray(ast.literal_eval(text), dtype=np.uint8).reshape(-1)
        side = math.isqrt(values.size)
        if side * side != values.size:
            raise ValueError(f"Contagem de pixels não quadrada: {values.size}")
        return Image.fromarray(values.reshape(side, side), mode="L").convert("RGB")

    # Carrega uma amostra e tenta a próxima quando há corrupção.
    def __getitem__(self, index):
        for step in range(len(self.records)):
            i = (index + step) % len(self.records)
            row = self.records[i]
            try:
                image = self._image(row)
                image = self.transform(image) if self.transform else image
                return (image, int(row["label"]))
            except (OSError, ValueError, SyntaxError, RuntimeError) as exc:
                self.corrupted_indices.add(i)
                logger.warning("Amostra corrompida (%s): %s", i, exc)
        raise RuntimeError("Todas as imagens tentadas estão corrompidas.")


# Cria os carregadores de treino e validação.
def create_dataloaders(config, train_dataset=None, val_dataset=None):
    config.validate()
    if train_dataset is None:
        train_dataset = EmotionFinetuneDataset(
            config.train_data_path,
            transform=build_transforms(config.image_size, True),
            dataset_format=config.dataset_format,
            label_scheme=config.label_scheme,
            image_root=config.csv_image_root,
            usage_filter=config.csv_usage_train,
            image_size=config.image_size,
        )
    if val_dataset is None:
        vp = config.val_data_path or config.train_data_path
        val_dataset = EmotionFinetuneDataset(
            vp,
            transform=build_transforms(config.image_size, False),
            dataset_format=config.dataset_format,
            label_scheme=config.label_scheme,
            image_root=config.csv_image_root,
            usage_filter=config.csv_usage_val if Path(vp).suffix.lower() == ".csv" else None,
            image_size=config.image_size,
        )
    args = dict(batch_size=config.batch_size, num_workers=config.num_workers, pin_memory=config.pin_memory)
    return (DataLoader(train_dataset, shuffle=True, **args), DataLoader(val_dataset, shuffle=False, **args))


assert [resolve_target_label(i, "fer") for i in (0, 3, 6, 4)] == [0, 1, 2, 3]
assert [resolve_target_label(i, "rafdb") for i in (6, 4, 7, 5)] == [0, 1, 2, 3]
assert [resolve_target_label(i, "affectnet") for i in (6, 1, 0, 2)] == [0, 1, 2, 3]
assert resolve_target_label("medo") is None and resolve_target_label("0_angry") == 0
print("Remapeamento FER/RAF-DB/AffectNet validado.")


## Seção 4 — Fine-tuning em duas fases

Fase 1: 70% do backbone congelado, AdamW e warmup linear. Fase 2: rede toda treinável, AdamW com LR `0.1×` backbone/`1×` cabeça e CosineAnnealingLR. A loss usa label smoothing. Cada época imprime Train Loss, Val Loss, Standard Acc, TTA Acc e Balanced Acc (TTA); o melhor checkpoint usa balanced accuracy TTA.

In [ ]:
# Métricas por época e fine-tuning progressivo em duas fases.
# Armazena as métricas de uma época de treino.
@dataclass
class EpochMetrics:
    train_loss: float
    val_loss: float
    standard_acc: float
    tta_acc: float
    balanced_acc_tta: float


# Resume o histórico e o melhor checkpoint do treino.
@dataclass
class TrainingHistory:
    epochs: list[dict[str, Any]]
    best_epoch: int
    best_balanced_acc: float
    best_checkpoint: str


# Combina predições originais e espelhadas horizontalmente.
def predict_batch_with_tta(model, images):
    logits = model(images)
    probs = torch.softmax(logits.float(), dim=1)
    flipped = torch.softmax(model(torch.flip(images, dims=[3])).float(), dim=1)
    return (logits, probs, 0.5 * (probs + flipped))


# Ativa autocast somente quando o treino usa GPU e AMP.
def _autocast(device, enabled):
    return torch.amp.autocast(device_type=device.type, enabled=bool(enabled and device.type == "cuda"))


# Executa uma época de otimização e calcula a perda média.
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, use_amp=True, grad_clip_norm=1.0):
    model.train()
    total = 0.0
    count = 0
    successful_steps = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()
        optimizer.zero_grad(set_to_none=True)
        with _autocast(device, use_amp):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        if grad_clip_norm is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip_norm)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if not scaler.is_enabled() or scaler.get_scale() >= old_scale:
            successful_steps += 1
        total += float(loss.detach()) * labels.size(0)
        count += labels.size(0)
    return (total / max(1, count), successful_steps)


# Orquestra warmup congelado, fine-tuning completo e checkpoints.
def run_finetuning(config, train_loader=None, val_loader=None, model=None):
    config.validate()
    seed_everything(config.seed)
    device = torch.device(config.device)
    if train_loader is None or val_loader is None:
        tr, va = create_dataloaders(config)
        train_loader = train_loader or tr
        val_loader = val_loader or va
    if model is None:
        ckpt = config.pretrained_checkpoint_path
        if ckpt is not None and (not Path(ckpt).is_file()):
            logger.warning("Checkpoint ausente; inicializando do zero.")
            ckpt = None
        model = load_pretrained_dscnn(
            ckpt,
            num_classes=config.num_classes,
            width_multiplier=config.width_multiplier,
            device=device,
            strict=ckpt is not None,
        )
    else:
        model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    scaler = torch.amp.GradScaler("cuda", enabled=bool(config.use_amp and device.type == "cuda"))
    output = Path(config.output_checkpoint_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    history_path = Path(config.history_json_path)
    history_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    best = -1.0
    best_epoch = 0
    for phase, first, last in [
        ("warmup", 1, config.warmup_epochs),
        ("finetune", config.warmup_epochs + 1, config.total_epochs),
    ]:
        if first > last:
            continue
        if phase == "warmup":
            cutoff = model.freeze_early_blocks(config.freeze_ratio)
            optimizer = torch.optim.AdamW(build_discriminative_param_groups(model, config, "warmup"))
            scheduler = torch.optim.lr_scheduler.LinearLR(
                optimizer, start_factor=0.2, end_factor=1.0, total_iters=max(1, last - first + 1)
            )
            logger.info("Fase 1: %d/%d blocos congelados.", cutoff, len(model.features))
        else:
            model.unfreeze_all()
            optimizer = torch.optim.AdamW(build_discriminative_param_groups(model, config, "finetune"))
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max(1, last - first + 1), eta_min=config.min_lr
            )
            logger.info("Fase 2: LR backbone %.2g, head %.2g.", config.lr_backbone, config.lr_head)
        for epoch in range(first, last + 1):
            train_loss, successful_steps = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, device, config.use_amp, config.grad_clip_norm
            )
            report = evaluate_model_with_tta(model, val_loader, criterion, device, config.use_amp)
            if successful_steps:
                scheduler.step()
            metrics = EpochMetrics(
                train_loss, report["val_loss"], report["standard_acc"], report["tta_acc"], report["balanced_acc_tta"]
            )
            row = {"epoch": epoch, "phase": phase, **asdict(metrics)}
            rows.append(row)
            logger.info(
                "Epoch %02d/%02d [%s] | Train Loss %.4f | Val Loss %.4f | Standard Acc %.4f | TTA Acc %.4f | Balanced Acc (TTA) %.4f",
                epoch,
                config.total_epochs,
                phase,
                train_loss,
                report["val_loss"],
                report["standard_acc"],
                report["tta_acc"],
                report["balanced_acc_tta"],
            )
            if report["balanced_acc_tta"] > best:
                best = report["balanced_acc_tta"]
                best_epoch = epoch
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "state_dict": model.state_dict(),
                        "epoch": epoch,
                        "best_balanced_acc": best,
                        "val_acc_tta": report["tta_acc"],
                        "val_acc_standard": report["standard_acc"],
                        "val_loss": report["val_loss"],
                        "classes": list(TARGET_CLASSES),
                        "config": config.to_dict(),
                    },
                    output,
                )
            history_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
    return asdict(TrainingHistory(rows, best_epoch, best, str(output)))


## Seção 5 — Avaliação com TTA e matriz de confusão

Calcula métricas standard/TTA, `classification_report` e matriz 4×4. `evaluate_checkpoint` recarrega o checkpoint e pode renderizar o gráfico de confusão.

In [ ]:
# Avaliação com TTA, relatório por classe e matriz de confusão.
# Agrupa métricas, previsões e matriz de confusão.
@dataclass
class EvaluationReport:
    val_loss: float
    standard_acc: float
    tta_acc: float
    balanced_acc_std: float
    balanced_acc_tta: float
    macro_f1_tta: float
    confusion_matrix: np.ndarray
    classification_report: str
    y_true: list[int]
    y_pred_std: list[int]
    y_pred_tta: list[int]

    def to_dict(self):
        return {
            **{
                k: getattr(self, k)
                for k in (
                    "val_loss",
                    "standard_acc",
                    "tta_acc",
                    "balanced_acc_std",
                    "balanced_acc_tta",
                    "macro_f1_tta",
                    "classification_report",
                    "y_true",
                    "y_pred_std",
                    "y_pred_tta",
                )
            },
            "confusion_matrix": self.confusion_matrix.tolist(),
        }


# Calcula métricas de validação com e sem TTA.
def evaluate_model_with_tta(model, dataloader, criterion=None, device="cpu", use_amp=False):
    device = torch.device(device)
    criterion = criterion or nn.CrossEntropyLoss()
    model.eval()
    total = 0.0
    count = 0
    truth = []
    pred = []
    pred_tta = []
    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).long()
            with _autocast(device, use_amp):
                logits, probs, tta = predict_batch_with_tta(model, images)
                loss = criterion(logits, labels)
            total += float(loss) * labels.size(0)
            count += labels.size(0)
            truth += labels.cpu().tolist()
            pred += probs.argmax(1).cpu().tolist()
            pred_tta += tta.argmax(1).cpu().tolist()
    if not truth:
        raise ValueError("Validação sem amostras.")
    labels = list(range(NUM_CLASSES))
    cm = confusion_matrix(truth, pred_tta, labels=labels)
    rep = classification_report(
        truth, pred_tta, labels=labels, target_names=list(TARGET_CLASSES), digits=4, zero_division=0
    )
    result = EvaluationReport(
        total / count,
        accuracy_score(truth, pred),
        accuracy_score(truth, pred_tta),
        balanced_accuracy_score(truth, pred),
        balanced_accuracy_score(truth, pred_tta),
        f1_score(truth, pred_tta, labels=labels, average="macro", zero_division=0),
        cm,
        rep,
        truth,
        pred,
        pred_tta,
    )
    return result.to_dict()


# Desenha a matriz de confusão por classe.
def plot_confusion_matrix(cm, class_names=TARGET_CLASSES):
    if plt is None:
        warnings.warn("matplotlib não disponível.")
        return None
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set(
        xticks=np.arange(4),
        yticks=np.arange(4),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted",
        ylabel="True",
        title="Confusion matrix (TTA)",
    )
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    for i in range(4):
        for j in range(4):
            ax.text(
                j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black"
            )
    fig.tight_layout()
    return fig


# Carrega o melhor checkpoint e imprime o relatório final.
def evaluate_checkpoint(checkpoint_path, val_loader, device="cpu", plot=True, width_multiplier=1.0):
    model = load_pretrained_dscnn(
        checkpoint_path, num_classes=NUM_CLASSES, width_multiplier=width_multiplier, device=device, strict=True
    )
    result = evaluate_model_with_tta(model, val_loader, device=device)
    print(result["classification_report"])
    print("Confusion matrix (rows=true, columns=predicted):")
    print(np.asarray(result["confusion_matrix"]))
    if plot and plt is not None:
        plot_confusion_matrix(np.asarray(result["confusion_matrix"]))
        plt.show()
    return result


## Seção 6 — Execução no Google Colab (Montagem do Google Drive)

Esta célula monta automaticamente o seu **Google Drive** (`/content/drive`) quando executada no Colab, carrega o modelo pré-treinado (`.pt`/`.pth`) e os datasets (`ImageFolder` ou `.csv`) diretamente do Drive e salva o melhor checkpoint ajustado de volta no Drive.

In [ ]:
# Montagem do Google Drive, caminhos editáveis e execução no Colab.
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Ambiente local detectado (fora do Google Colab).")
DRIVE_ROOT = Path("/content/drive/MyDrive")
CHECKPOINT_PATH = DRIVE_ROOT / "models/dscnn_pretrained.pt"
TRAIN_PATH = DRIVE_ROOT / "datasets/emotions/train"
VAL_PATH = DRIVE_ROOT / "datasets/emotions/val"
OUTPUT_PATH = DRIVE_ROOT / "models/best_dscnn_finetuned.pt"
if TRAIN_PATH.exists():
    config = FineTuneConfig(
        pretrained_checkpoint_path=CHECKPOINT_PATH if CHECKPOINT_PATH.is_file() else None,
        train_data_path=TRAIN_PATH,
        val_data_path=VAL_PATH if VAL_PATH.exists() else None,
        output_checkpoint_path=OUTPUT_PATH,
        history_json_path=OUTPUT_PATH.with_name("finetune_history.json"),
        dataset_format="auto",
        label_scheme="auto",
        image_size=48,
        width_multiplier=1.0,
        batch_size=64,
        warmup_epochs=5,
        total_epochs=30,
        lr_head=0.0003,
        lr_backbone_factor=0.1,
        weight_decay=0.01,
        label_smoothing=0.1,
        device=str(RUNTIME_DEVICE),
        use_amp=RUNTIME_DEVICE.type == "cuda",
        num_workers=2,
        pin_memory=True,
    )
    train_loader, val_loader = create_dataloaders(config)
    print(
        f"Train: {len(train_loader.dataset)} amostras (descartadas: {train_loader.dataset.discarded_count}) | Val: {len(val_loader.dataset)} amostras (descartadas: {val_loader.dataset.discarded_count})"
    )
    training_result = run_finetuning(config, train_loader, val_loader)
    print(
        f"Melhor checkpoint salvo em: {training_result['best_checkpoint']} | Best Balanced Acc (TTA): {training_result['best_balanced_acc']:.4f}"
    )
    final_report = evaluate_checkpoint(
        training_result["best_checkpoint"],
        val_loader,
        device=RUNTIME_DEVICE,
        plot=plt is not None,
        width_multiplier=config.width_multiplier,
    )
else:
    print(
        f"[AVISO] Caminho de treino não encontrado em: {TRAIN_PATH}\nAjuste DRIVE_ROOT, CHECKPOINT_PATH, TRAIN_PATH e VAL_PATH acima para os caminhos do seu Google Drive."
    )
